## HWPX 파싱

In [2]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w5" / "day02"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [3]:
if not SAMPLES.is_dir():
    print('재료 폴더가 없습니다.')
else:
    all_files = [p for p in SAMPLES.rglob('*') if p.is_file()]
    top_files = [p for p in SAMPLES.iterdir() if p.is_file()]
    sub_files = [p for p in (SAMPLES / '_formats').rglob('*') if p.is_file()]
    practice = sorted(SAMPLES.glob('*.hwpx'))
    print(f'재료 파일 : {len(all_files)}개')
    print(f'  samples 바로 아래 : {len(top_files)}개 · _formats 안 : {len(sub_files)}개')
    print(f'  그중 한글 문서 연습 재료 : {len(practice)}건   ← 오늘 하루 이 셋을 읽습니다')


재료 파일 : 30개
  samples 바로 아래 : 24개 · _formats 안 : 6개
  그중 한글 문서 연습 재료 : 3건   ← 오늘 하루 이 셋을 읽습니다


In [4]:
from app.rag.local_parsers import parse_docx
TARGET = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
doc = parse_docx(TARGET)
print(f'블록 수    : {len(doc.blocks)}')
print(f'표 수      : {doc.table_count}')
print(f'page_count : {doc.page_count}')
first = doc.blocks[0]
print(f'첫 블록 kind : {first.kind}')
print(f'첫 블록 text : {first.text}')


블록 수    : 233
표 수      : 7
page_count : 1
첫 블록 kind : 조항
첫 블록 text : 일반


In [5]:
from app.rag.local_parsers import parse_pdf

NAIVE = {'.docx': parse_docx, '.pdf': parse_pdf}

read_ok, skipped = (0, 0)
for p in sorted(SAMPLES.iterdir()):
    if not p.is_file():
        continue
    handler = NAIVE.get(p.suffix.lower())
    if handler is None:
        skipped += 1
        continue
    handler(p)
    read_ok += 1
print(f'훑은 파일   : {read_ok + skipped}건')
print(f'읽힘        : {read_ok}건')
print(f'조용히 빠짐 : {skipped}건')


훑은 파일   : 24건
읽힘        : 20건
조용히 빠짐 : 4건


In [6]:
ZIP_MAGIC = b'PK\x03\x04'
heads = [p.read_bytes()[:4] for p in sorted(SAMPLES.glob('*.hwpx'))]
for n, head in enumerate(heads, 1):
    print(f'연습 재료 {n} : 앞 4바이트 {head!r}  →  ZIP 인가 {head == ZIP_MAGIC}')
print('세 건 모두 ZIP :', all((h == ZIP_MAGIC for h in heads)))

연습 재료 1 : 앞 4바이트 b'PK\x03\x04'  →  ZIP 인가 True
연습 재료 2 : 앞 4바이트 b'PK\x03\x04'  →  ZIP 인가 True
연습 재료 3 : 앞 4바이트 b'PK\x03\x04'  →  ZIP 인가 True
세 건 모두 ZIP : True


In [7]:
import zipfile

HWPX = next(SAMPLES.glob("DOC-HR-014_*_v2.0.hwpx"))
print("가져온 파일: ", HWPX.name)  
print()

with zipfile.ZipFile(HWPX) as zf:
    names = zf.namelist() 

print("항목 수 : ", len(names))
for n in names:
    print(" ", n)

가져온 파일:  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx

항목 수 :  10
  mimetype
  version.xml
  META-INF/container.xml
  META-INF/container.rdf
  META-INF/manifest.xml
  Contents/content.hpf
  Contents/header.xml
  Contents/section0.xml
  settings.xml
  Preview/PrvText.txt


In [8]:
with zipfile.ZipFile(HWPX) as zf:
    raw_mime = zf.read('mimetype')
    info = zf.getinfo('mimetype')
    ways = {zf.getinfo(n).compress_type for n in names}
print('mimetype 내용 :', raw_mime.decode('ascii'))
print('바이트 수     :', len(raw_mime))
print('압축 방식     :', info.compress_type, '(0 = 압축한 것이 아니라 그대로 넣었다)')
print('열 개 전체의 압축 방식 :', ways)


mimetype 내용 : application/hwp+zip
바이트 수     : 19
압축 방식     : 0 (0 = 압축한 것이 아니라 그대로 넣었다)
열 개 전체의 압축 방식 : {0}


In [9]:
import re

with zipfile.ZipFile(HWPX) as zf:
    sections = sorted(n for n in zf.namelist() if re.match(r"Contents/section\d+\.xml$", n))

print("본문파일 : ", sections)
print("개수 : ", len(sections))

본문파일 :  ['Contents/section0.xml']
개수 :  1


In [11]:
with zipfile.ZipFile(HWPX) as zf:
    raw = zf.read(sections[0])

print("type : ", type(raw).__name__)
print("길이 : ", f"{len(raw):,}", "바이트")
print(raw[:200].decode("utf-8"))

type :  bytes
길이 :  190,833 바이트
<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<hs:sec xmlns:hs="http://www.hancom.co.kr/hwpml/2011/section" xmlns:hp="http://www.hancom.co.kr/hwpml/2011/paragraph" xmlns:hc="http://www.hanco
